In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy
device="cuda"

In [ ]:
## Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_query = nn.Linear(d_model, d_model)
        self.W_key = nn.Linear(d_model, d_model)
        self.W_value = nn.Linear(d_model, d_model)
        self.W_output = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, query, key, value, mask=None):
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
        
        attention_probabilities = torch.softmax(attention_scores, dim=-1)
        output = torch.matmul(attention_probabilities, value)
        return output

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, query, key, value, mask=None):
        query = self.split_heads(self.W_query(query))
        key = self.split_heads(self.W_key(key))
        value = self.split_heads(self.W_value(value))
        
        attention_output = self.scaled_dot_product_attention(query, key, value, mask)
        output = self.W_output(self.combine_heads(attention_output))
        return output

In [ ]:
## Position-Wise Feed Forward
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

## Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length, device):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_seq_length, d_model, device=device)
        position = torch.arange(0, max_seq_length, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
## Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attention_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attention_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

## Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        self_attention_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attention_output))
        
        cross_attention_output = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attention_output))
        
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [ ]:
#Model
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout, device):
        super(Transformer, self).__init__()
        self.device = device
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length, device)
        
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2).to(self.device)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3).to(self.device)
        seq_length = tgt.size(1)
        
        no_peak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length, device=self.device), diagonal=1)).bool()
        tgt_mask = tgt_mask & no_peak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))
        
        encoder_output = src_embedded
        for layer in self.encoder_layers:
            encoder_output = layer(encoder_output, src_mask)
            
        decoder_output = tgt_embedded
        for layer in self.decoder_layers:
            decoder_output = layer(decoder_output, encoder_output, src_mask, tgt_mask)
            
        output = self.fc_out(decoder_output)
        return output

In [ ]:
#hyperparameters
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

#Model
transformer = Transformer(
    src_vocab_size, 
    tgt_vocab_size, 
    d_model, 
    num_heads, 
    num_layers, 
    d_ff, 
    max_seq_length, 
    dropout, 
    device
).to(device)

#Sample Data
source_data = torch.randint(1, src_vocab_size, (64, max_seq_length)).to(device)
target_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length)).to(device)

# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(
    transformer.parameters(), 
    lr=0.0003, 
    betas=(0.9, 0.98), 
    eps=1e-9
)

In [ ]:
transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    decoder_input = target_data[:, :-1]
    expected_output = target_data[:, 1:]
    output = transformer(source_data, decoder_input)
    loss = criterion(
        output.contiguous().view(-1, tgt_vocab_size), 
        expected_output.contiguous().view(-1)
    )
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch + 1} | Loss: {loss.item():.4f}")

Epoch:1  Loss: 8.683809280395508
Epoch:2  Loss: 8.51290512084961
Epoch:3  Loss: 8.396591186523438
Epoch:4  Loss: 8.214051246643066
Epoch:5  Loss: 8.069912910461426
Epoch:6  Loss: 7.9469780921936035
Epoch:7  Loss: 7.815495491027832
Epoch:8  Loss: 7.7025909423828125
Epoch:9  Loss: 7.474428176879883
Epoch:10  Loss: 7.309690475463867
Epoch:11  Loss: 7.155665397644043
Epoch:12  Loss: 7.007244110107422
Epoch:13  Loss: 6.849619388580322
Epoch:14  Loss: 6.68565034866333
Epoch:15  Loss: 6.536060810089111
Epoch:16  Loss: 6.400396823883057
Epoch:17  Loss: 6.274723529815674
Epoch:18  Loss: 6.13687801361084
Epoch:19  Loss: 5.978294372558594
Epoch:20  Loss: 5.8505706787109375
Epoch:21  Loss: 5.677334785461426
Epoch:22  Loss: 5.555316925048828
Epoch:23  Loss: 5.444845199584961
Epoch:24  Loss: 5.289299964904785
Epoch:25  Loss: 5.159487247467041
Epoch:26  Loss: 5.03914737701416
Epoch:27  Loss: 4.911040782928467
Epoch:28  Loss: 4.802249908447266
Epoch:29  Loss: 4.686496257781982
Epoch:30  Loss: 4.530769

In [ ]:
transformer.eval()

# Validation Data Generation
validation_source_data = torch.randint(1, src_vocab_size, (64, max_seq_length)).to(device)
validation_target_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length)).to(device)

# Validation
with torch.no_grad():
    validation_decoder_input = validation_target_data[:, :-1]
    validation_expected_output = validation_target_data[:, 1:]
    validation_output = transformer(validation_source_data, validation_decoder_input)
    validation_loss = criterion(
        validation_output.contiguous().view(-1, tgt_vocab_size), 
        validation_expected_output.contiguous().view(-1)
    )
    
    print(f"Validation Loss: {validation_loss.item():.4f}")

Validation Loss: 9.64944076538086
